In [1]:
import sys
import os
import pickle

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Add project root to path so utils.py is importable from notebooks/
sys.path.append(os.path.abspath('..'))
from utils import add_holiday_features, add_lag_features, add_rolling_features

# ── Config ──────────────────────────────────────────
TARGET_COUNTRIES = ['HU', 'DE', 'FR', 'IT']
LAG_HOURS        = [1, 24, 168]   # 1h, 1 day, 1 week
ROLLING_WINDOWS  = [24, 168]      # 1 day, 1 week
LOOKBACK         = 24             # LSTM input window (hours)
OUTPUT_DIR       = '../data/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK')

Imports OK


## 1. Load Cleaned Data

In [2]:
DATA_PATH = '../data/processed/clean.parquet'

df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith('.parquet') else pd.read_csv(
    DATA_PATH, index_col='dateutc', parse_dates=True
)

# Filter to target countries and sort by time
df = df[df['countrycode'].isin(TARGET_COUNTRIES)].sort_index().copy()

print(f'Rows: {len(df):,}')
print(f'Countries: {sorted(df["countrycode"].unique())}')
print(f'Period: {df.index.min()} → {df.index.max()}')
df.head()

Rows: 236,572
Countries: ['DE', 'FR', 'HU', 'IT']
Period: 2019-01-01 00:00:00+00:00 → 2025-09-30 23:00:00+00:00


,dateshort,timefrom,timeto,countrycode,value,value_scaleto100,year
dateutc,,,,,,,
2019-01-01 00:00:00+00:00,2019-01-01,00:00:00,01:00:00,DE,41653.9575,41653.9575,2019
2019-01-01 00:00:00+00:00,2019-01-01,00:00:00,01:00:00,HU,3985.6200,3985.6200,2019
2019-01-01 00:00:00+00:00,2019-01-01,00:00:00,01:00:00,FR,60301.0000,60301.0000,2019
2019-01-01 00:00:00+00:00,2019-01-01,00:00:00,01:00:00,IT,22850.0000,22850.0000,2019
2019-01-01 01:00:00+00:00,2019-01-01,01:00:00,02:00:00,DE,40113.5800,40113.5800,2019


## 2. Time-based Features
Cyclic encoding (sin/cos) ensures that hour 23 → 0 is treated as continuous by the model.

In [3]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    idx = df.index.tz_convert('UTC') if df.index.tzinfo else df.index

    df['hour']       = idx.hour
    df['dayofweek']  = idx.dayofweek   # 0=Monday, 6=Sunday
    df['month']      = idx.month
    df['quarter']    = idx.quarter
    df['dayofyear']  = idx.dayofyear
    df['is_weekend'] = (idx.dayofweek >= 5).astype(int)

    # Cyclic encoding
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']      / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']      / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']     / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']     / 12)

    return df

df = add_time_features(df)
print('Time features added:', ['hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos','is_weekend'])
df[['hour','hour_sin','hour_cos','dayofweek','is_weekend']].head()

Time features added: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']


,hour,hour_sin,hour_cos,dayofweek,is_weekend
dateutc,,,,,
2019-01-01 00:00:00+00:00,0,0.000000,1.000000,1,0
2019-01-01 00:00:00+00:00,0,0.000000,1.000000,1,0
2019-01-01 00:00:00+00:00,0,0.000000,1.000000,1,0
2019-01-01 00:00:00+00:00,0,0.000000,1.000000,1,0
2019-01-01 01:00:00+00:00,1,0.258819,0.965926,1,0


In [4]:
df = add_holiday_features(df)

# Quick check: how many holiday hours per country?
df.groupby('countrycode')['is_holiday'].sum().rename('holiday_hours')

countrycode
DE    1440
FR    1776
HU    2376
IT    2040
Name: holiday_hours, dtype: int64

In [5]:
df = add_lag_features(df, lags=LAG_HOURS)

df[['countrycode', 'value', 'load_lag_1h', 'load_lag_24h', 'load_lag_168h']].head(10)

,countrycode,value,load_lag_1h,load_lag_24h,load_lag_168h
dateutc,,,,,
2019-01-01 00:00:00+00:00,DE,41653.9575,NaN,NaN,NaN
2019-01-01 00:00:00+00:00,HU,3985.6200,NaN,NaN,NaN
2019-01-01 00:00:00+00:00,FR,60301.0000,NaN,NaN,NaN
2019-01-01 00:00:00+00:00,IT,22850.0000,NaN,NaN,NaN
2019-01-01 01:00:00+00:00,DE,40113.5800,41653.9575,NaN,NaN
2019-01-01 01:00:00+00:00,IT,21600.0000,22850.0000,NaN,NaN
2019-01-01 01:00:00+00:00,HU,3732.5325,3985.6200,NaN,NaN
2019-01-01 01:00:00+00:00,FR,58540.0000,60301.0000,NaN,NaN
2019-01-01 02:00:00+00:00,HU,3554.1075,3732.5325,NaN,NaN


In [6]:
df = add_rolling_features(df, windows=ROLLING_WINDOWS)

df[['countrycode', 'value', 'load_roll_24h_mean', 'load_roll_24h_std']].head(10)

,countrycode,value,load_roll_24h_mean,load_roll_24h_std
dateutc,,,,
2019-01-01 00:00:00+00:00,DE,41653.9575,NaN,NaN
2019-01-01 00:00:00+00:00,HU,3985.6200,NaN,NaN
2019-01-01 00:00:00+00:00,FR,60301.0000,NaN,NaN
2019-01-01 00:00:00+00:00,IT,22850.0000,NaN,NaN
2019-01-01 01:00:00+00:00,DE,40113.5800,41653.95750,NaN
2019-01-01 01:00:00+00:00,IT,21600.0000,22850.00000,NaN
2019-01-01 01:00:00+00:00,HU,3732.5325,3985.62000,NaN
2019-01-01 01:00:00+00:00,FR,58540.0000,60301.00000,NaN
2019-01-01 02:00:00+00:00,HU,3554.1075,3859.07625,178.959887


In [7]:
before = len(df)
df = df.dropna()
print(f'Rows before: {before:,} → after dropna: {len(df):,} (dropped {before - len(df):,})')

Rows before: 236,572 → after dropna: 200,869 (dropped 35,703)


In [8]:
FEATURE_COLS = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend', 'is_holiday',
    'load_lag_1h', 'load_lag_24h', 'load_lag_168h',
    'load_roll_24h_mean', 'load_roll_24h_std',
    'load_roll_168h_mean', 'load_roll_168h_std',
    'value'  # target column — scaled last
]

scalers = {}
df_scaled = df.copy()

for cc in TARGET_COUNTRIES:
    mask = df_scaled['countrycode'] == cc
    subset = df_scaled.loc[mask, FEATURE_COLS]
    sc = MinMaxScaler()
    df_scaled.loc[mask, FEATURE_COLS] = sc.fit_transform(subset)
    scalers[cc] = sc

# Save scalers for inference later
with open(f'{OUTPUT_DIR}/scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print(f'Scalers saved → {OUTPUT_DIR}/scalers.pkl')
df_scaled[FEATURE_COLS].describe().round(3)

Scalers saved → ../data/processed/scalers.pkl


,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend,is_holiday,load_lag_1h,load_lag_24h,load_lag_168h,load_roll_24h_mean,load_roll_24h_std,load_roll_168h_mean,load_roll_168h_std,value
count,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000,200869.000
mean,0.500,0.500,0.500,0.474,0.507,0.480,0.286,0.032,0.452,0.452,0.453,0.466,0.436,0.425,0.470,0.452
std,0.354,0.354,0.363,0.372,0.356,0.351,0.452,0.175,0.191,0.191,0.191,0.202,0.164,0.230,0.182,0.191
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.146,0.146,0.099,0.000,0.250,0.067,0.000,0.000,0.302,0.302,0.303,0.299,0.313,0.230,0.326,0.302
50%,0.500,0.500,0.500,0.357,0.500,0.500,0.000,0.000,0.443,0.443,0.444,0.473,0.439,0.411,0.462,0.443
75%,0.854,0.854,0.901,0.802,0.933,0.750,1.000,0.000,0.596,0.596,0.596,0.612,0.556,0.597,0.608,0.596
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


In [9]:
def build_sequences(df_country: pd.DataFrame, feature_cols: list, lookback: int = LOOKBACK):
    """Convert a country DataFrame into LSTM-ready sequences."""
    data   = df_country[feature_cols].values
    target = df_country['value'].values
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])
        y.append(target[i])
    return np.array(X), np.array(y)


def temporal_split(X, y, val_ratio=0.1, test_ratio=0.1):
    """Time-ordered split — no shuffling."""
    n = len(X)
    test_start = int(n * (1 - test_ratio))
    val_start  = int(n * (1 - test_ratio - val_ratio))
    return {
        'X_train': X[:val_start],          'y_train': y[:val_start],
        'X_val':   X[val_start:test_start], 'y_val':   y[val_start:test_start],
        'X_test':  X[test_start:],          'y_test':  y[test_start:],
    }


datasets = {}
for cc in TARGET_COUNTRIES:
    df_cc = df_scaled[df_scaled['countrycode'] == cc].copy()
    if len(df_cc) < LOOKBACK + 100:
        print(f'⚠ {cc}: not enough data, skipping')
        continue
    X, y = build_sequences(df_cc, FEATURE_COLS)
    splits = temporal_split(X, y)
    datasets[cc] = splits
    print(f'{cc} → train: {len(splits["X_train"]):,} | val: {len(splits["X_val"]):,} | test: {len(splits["X_test"]):,}')

print(f'\nX shape example ({list(datasets.keys())[0]}): {list(datasets.values())[0]["X_train"].shape}')

HU → train: 40,164 | val: 5,021 | test: 5,021
DE → train: 40,166 | val: 5,021 | test: 5,021
FR → train: 40,120 | val: 5,015 | test: 5,016
IT → train: 40,166 | val: 5,021 | test: 5,021

X shape example (HU): (40164, 24, 16)


In [10]:
# Save featured DataFrame
df_scaled.to_parquet(f'{OUTPUT_DIR}/featured_df.parquet')

# Save datasets dict (numpy arrays per country)
np.save(f'{OUTPUT_DIR}/datasets.npy', datasets, allow_pickle=True)

print(f'Saved to {OUTPUT_DIR}/')
print(f'  featured_df.parquet  →  full scaled DataFrame')
print(f'  datasets.npy         →  X/y splits per country')
print(f'  scalers.pkl          →  MinMaxScaler per country')

Saved to ../data/processed/
  featured_df.parquet  →  full scaled DataFrame
  datasets.npy         →  X/y splits per country
  scalers.pkl          →  MinMaxScaler per country
